# AdaBoost Regression - California Housing Dataset


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Load Dataset

In [ ]:
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = pd.Series(housing.target, name='MedHouseVal')

df = X.copy()
df['MedHouseVal'] = y
print(df.shape)
df.head()

## EDA

In [ ]:
print(df.info())
print(df.describe())

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(y, kde=True, color='#023e8a')
plt.title('House Value Distribution')
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='Blues', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

## Hyperparameter Tuning

In [ ]:
# Test different n_estimators
estimators = [10, 50, 100, 150, 200]
scores = []
for n in estimators:
    reg = AdaBoostRegressor(n_estimators=n, random_state=42)
    score = cross_val_score(reg, X, y, cv=3, scoring='r2').mean()
    scores.append(score)

plt.figure(figsize=(8, 4))
plt.plot(estimators, scores, marker='o', color='#023e8a')
plt.xlabel('n_estimators')
plt.ylabel('CV R² Score')
plt.title('Effect of n_estimators on R² Score')
plt.show()

## Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train size:', X_train.shape)
print('Test size:', X_test.shape)

## Train Model

In [ ]:
base_estimator = DecisionTreeRegressor(max_depth=3)
model = AdaBoostRegressor(
    estimator=base_estimator,
    n_estimators=100,
    learning_rate=1.0,
    loss='linear',
    random_state=42
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

## Evaluation

In [ ]:
print('MAE:', mean_absolute_error(y_test, y_pred))
print('MSE:', mean_squared_error(y_test, y_pred))
print('R2 Score:', r2_score(y_test, y_pred))

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_test[:200], y_pred[:200], alpha=0.5, color='#023e8a')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Actual vs Predicted')
plt.show()

## Feature Importance

In [ ]:
importance = pd.DataFrame({
    'Feature': housing.feature_names,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(x='Importance', y='Feature', data=importance, palette='Blues_r')
plt.title('Feature Importance')
plt.show()

## Cross Validation

In [ ]:
cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')
print('CV R2 Scores:', cv_scores)
print('Mean CV R2 Score:', cv_scores.mean())

plt.figure(figsize=(8, 4))
plt.bar(range(1, 6), cv_scores, color='#023e8a')
plt.axhline(cv_scores.mean(), color='red', linestyle='--', label=f'Mean: {cv_scores.mean():.4f}')
plt.xlabel('Fold')
plt.ylabel('R² Score')
plt.title('5-Fold Cross Validation R² Scores')
plt.legend()
plt.show()

## Actual vs Predicted Table

In [ ]:
comparison = pd.DataFrame({
    'Actual': y_test.values[:20],
    'Predicted': y_pred[:20]
})
comparison